# Tydzień 1: Definicja problemu

Projekt: **Automatyczne streszczanie artykułów**. Zbiór danych: **CNN/DailyMail 3.0.0 (Hugging Face)**.

Notatnik obejmuje zadania Z1 (sprawdzenie zbioru danych) i Z4 (baseline Lead-3).

## Z1. Sprawdzenie zbioru danych

Cel: upewnić się, że dane wczytują się poprawnie, mają oczekiwany rozmiar i strukturę, oraz zobaczyć, jak wyglądają artykuły i streszczenia.

### 1.1 Wczytanie danych

Przy pierwszym uruchomieniu pobiera się ok. 840 MB. Kolejne uruchomienia korzystają z pamięci podręcznej (`~/.cache/huggingface`).

In [11]:
%pip install evaluate rouge_score nltk

  Using cached rouge_score-0.1.2-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.9 MB/s eta 0:00:00 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import load_dataset
import pandas as pd
import evaluate
import nltk

nltk.download("punkt_tab")
rouge = evaluate.load("rouge")

ds = load_dataset("abisee/cnn_dailymail", "3.0.0")
ds

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1010)>


DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

### 1.2 Rozmiary podziałów

Porównanie z wartościami z opisu zbioru na Hugging Face.

In [3]:
oczekiwane = {"train": 287_113, "validation": 13_368, "test": 11_490}

rozmiary = pd.DataFrame({
    "liczba przykładów": {split: ds[split].num_rows for split in ds},
    "oczekiwane": oczekiwane,
})
rozmiary["zgodne"] = rozmiary["liczba przykładów"] == rozmiary["oczekiwane"]
rozmiary

,liczba przykładów,oczekiwane,zgodne
train,287113,287113,True
validation,13368,13368,True
test,11490,11490,True


### 1.3 Kolumny i typy danych

In [4]:
ds["train"].features

{'article': Value('string'),
 'highlights': Value('string'),
 'id': Value('string')}

### 1.4 Braki danych i puste teksty

In [5]:
test = ds["test"].to_pandas()

pd.DataFrame({
    "braki (NaN)": test.isna().sum(),
    "puste teksty": (test.astype(str).apply(lambda kol: kol.str.strip()) == "").sum(),
    "duplikaty": test.apply(lambda kol: kol.duplicated().sum()),
})

,braki (NaN),puste teksty,duplikaty
article,0,0,2
highlights,0,0,41
id,0,0,0


In [6]:
test[test["highlights"].duplicated(keep=False)].sort_values("highlights")

,article,highlights,id
2160,The omens are looking good. After taking the b...,AP McCoy wins second feature race at Grand Nat...,05ccf000bb7aa7797c92e21ce54eaf94478b359a
3426,AP McCoy landed the second win of his last Ain...,AP McCoy wins second feature race at Grand Nat...,9c1adfd409948f56e533435079884d8232a9c087
3701,Sunderland boss Dick Advocaat has insisted it ...,Adam Johnson charged with three offences of se...,f1beee2435f59b05e0da484c7a1c167709412d6d
3732,Sunderland will not suspend Adam Johnson after...,Adam Johnson charged with three offences of se...,2f66bfae4f6c7bc609e9acadb660a6499240c6bc
2559,Andy Murray has shown nerves of steel on the t...,Andy Murray is getting married to Kim Sears in...,0b0e4349b7947078087407f6a0fd2b465de31cc6
...,...,...,...
7698,Defiant Nigel Farage today insisted he did not...,Ukip leader Nigel Farage risks alienating thos...,5e7b53548bb3a7998b9ed67d23bb3507ecef8ee0
9212,Defiant Nigel Farage today insisted he did not...,Ukip leader Nigel Farage risks alienating thos...,26252d601e52ff5ffabde202bc4883ee24b4ba49
9458,Nigel Farage was booed by the audience of last...,Ukip leader Nigel Farage risks alienating thos...,964385e302b05af7a2a4e7585d65d951efb2245d
4628,Pictures of Suge Knight showing he had virtual...,WARNING: GRAPHIC CONTENT .\nFootage was releas...,51a4ce414da2c1440eacaf53999a55e7e6116487


In [10]:
for i in [7698, 9212, 4628]:
    print(i, test.loc[i, "article"][:300])
    print("HL:", test.loc[i, "highlights"])
    print()


7698 Defiant Nigel Farage today insisted he did not 'lose my rag' after rounding on the audience of a live TV debate for being too left wing. The Ukip leader was booed by voters at Westminster's Methodist Central Hall as he faced claims he blamed all of Britain's problems on migrants. Pollster ICM, which
HL: Ukip leader Nigel Farage risks alienating those watching debate last night .
Complains of 'remarkable audience even by Left-wing standards of BBC'
Comments on housing pressure due to immigration greeted with mutters .
David Dimbleby says independent polling firm chose 'balanced' audience .

9212 Defiant Nigel Farage today insisted he did not 'lose my rag' after rounding on the audience of a live TV debate for being too left wing. The Ukip leader was booed by voters at Westminster's Methodist Central Hall as he faced claims he blamed all of Britain's problems on migrants. Pollster ICM, which
HL: Ukip leader Nigel Farage risks alienating those watching debate last night .
Complains o

### 1.5 Przykładowe artykuły i streszczenia

In [7]:
for i in range(5):
    przyklad = ds["test"][i]
    print(f"=== Przykład {i + 1} (id: {przyklad['id']}) ===")
    print("ARTYKUŁ (początek):")
    print(przyklad["article"][:500], "...")
    print("\nSTRESZCZENIE (highlights):")
    print(przyklad["highlights"])
    print()

=== Przykład 1 (id: f001ec5c4704938247d27a44948eebb37ae98d01) ===
ARTYKUŁ (początek):
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, includin ...

STRESZCZENIE (highlights):
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

=== Przykład 2 (id: 230c522854991d053fe98a718b1defa077a8efef) ===
ARTYKUŁ (początek):
(CNN)Never mind cats having nine lives. A stray pooch in Was

### 1.6 Pierwsze spojrzenie na długości

Szybki podgląd w słowach. Dokładne rozkłady, w tym długości w tokenach, będą w EDA (tydzień 2).

In [8]:
dlugosci = pd.DataFrame({
    "artykuł (słowa)": test["article"].str.split().str.len(),
    "streszczenie (słowa)": test["highlights"].str.split().str.len(),
})
dlugosci.describe().round(0)

,artykuł (słowa),streszczenie (słowa)
count,11490.0,11490.0
mean,684.0,55.0
std,348.0,23.0
min,55.0,9.0
25%,419.0,40.0
50%,613.0,51.0
75%,876.0,64.0
max,1954.0,564.0


In [9]:
zrodlo = test["article"].str.contains(r"\(CNN\)").map({True: "CNN", False: "Daily Mail"})
zrodlo.value_counts(normalize=True).round(3)

article
Daily Mail    0.907
CNN           0.093
Name: proportion, dtype: float64

### 1.7 Wnioski z Z1

- Rozmiary podziałów zgadzają się z opisem zbioru. 
- 78 artykułów (37 grup) ma powtarzające się streszczenia. W większości to różne artykuły o tym samym wydarzeniu, dla których Daily Mail użył tych samych punktów (np. AP McCoy, Adam Johnson). Niektóre artykuły są niemal identyczne (Nigel Farage, wiersze 7698 i 9212). Część streszczeń zawiera dopiski redakcyjne, np. „WARNING: GRAPHIC CONTENT”. To szum w danych do uwzględnienia w preprocessingu. Artykuły w wierszach 7698 i 9212 (oraz 987 i 988) są identyczne i różnią się tylko id. Streszczenia mają typowo 4 punkty rozdzielone znakiem nowej linii, ale zdarzają się dłuższe (do 39 punktów) i takie z dopiskami redakcyjnymi („WARNING: GRAPHIC CONTENT”). Zbioru testowego nie czyszczę, żeby wyniki były porównywalne z literaturą. Czyszczenie dotyczy tylko zbioru treningowego.
- Streszczenia to punkty (highlights): zwykle 4 krótkie zdania (mediana), każde w osobnej linii. Zdarzają się dłuższe, do 39 punktów. Przed kropką jest spacja (last June .).
- Średnio słów ma artykuł 684, a streszczenie 55.
- 0.093 artykułów pochodzi z CNN, a 0.907 z Daily Mail.

## Z2. Typ zadania

- Typ zadania ML: Generowanie 
- uczenie nadzorowane. Wejście article, etykieta, highlights.
- Architektura: Seq2Seq z Attention 
- Nie można tu użyć accuracy. Accuracy wymaga dokładnego trafienia w poprawną odpowiedź. Ten sam artykuł można jednak dobrze streścić na wiele sposobów, a model prawie nigdy nie napisze streszczenia identycznego ze streszczeniem redakcji, więc accuracy byłoby bliskie zera nawet dla dobrego modelu. Dlatego używam ROUGE, które mierzy, ile słów i fraz (n-gramów) pokrywa się ze streszczeniem referencyjnym.

## Z3. Pytanie badawcze

**Pytanie główne:**
Czy model `facebook/bart-base`, fine-tunowany na podzbiorze CNN/DailyMail na laptopie (MacBook M2 Pro), osiągnie wyższy ROUGE-2 niż baseline Lead-3 i jak daleko będzie od modelu `bart-large-cnn`?

**Pytania pomocnicze:**
1. Czy model gorzej streszcza długie artykuły (powyżej 1024 tokenów), które są obcinane, niż krótkie?
2. Czy model streszcza cały artykuł, czy głównie jego początek? (mierzone tym, z której części artykułu pochodzą słowa streszczenia)
3. Jak często model podaje fakty, liczby lub nazwiska, których nie ma w artykule? (ręczna ocena 30–50 streszczeń)

## Z4. Baseline Lead-3

Lead-3 jako streszczenie bierze 3 pierwsze zdania artykułu. Nie uczy się niczego, a mimo to jest mocnym baseline'em, bo w artykułach prasowych najważniejsze informacje są na początku (lead bias).


In [2]:
przyklad = ds["test"][0]
lead3 = nltk.sent_tokenize(przyklad["article"])[:3]

print("LEAD-3:")
print("\n".join(lead3))
print("\nSTRESZCZENIE REDAKCJI:")
print(przyklad["highlights"])


LEAD-3:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories.
The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based.
The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014."

STRESZCZENIE REDAKCJI:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [3]:
lead3 = ["\n".join(nltk.sent_tokenize(a)[:3]) for a in ds["test"]["article"]]
refs = ["\n".join(nltk.sent_tokenize(h)) for h in ds["test"]["highlights"]]

wynik = rouge.compute(predictions=lead3, references=refs, use_stemmer=True)

pd.DataFrame({
    "Lead-3 (mój wynik)": {k: round(v * 100, 2) for k, v in wynik.items()},
    "Lead-3 (literatura)": {"rouge1": 40.34, "rouge2": 17.70, "rougeL": None, "rougeLsum": 36.57},
    "bart-large-cnn (sufit)": {"rouge1": 44.16, "rouge2": 21.28, "rougeL": None, "rougeLsum": 40.90},
})


,Lead-3 (mój wynik),Lead-3 (literatura),bart-large-cnn (sufit)
rouge1,40.05,40.34,44.16
rouge2,17.49,17.70,21.28
rougeL,25.02,NaN,NaN
rougeLsum,36.38,36.57,40.90


### Wnioski z Z4
- Lead-3 osiągnął ROUGE-2 = 17,49. W literaturze jest 17,70, czyli różnica 0,21 punktu. Wynik jest zgodny, więc ewaluacja działa poprawnie.
- Lead-3 dzieli od bart-large-cnn 3,79 punktu ROUGE-2. Cel projektu (19,0) oznacza odrobienie ok. 40% tej różnicy.
- Lead-3 jest ponad 2 razy dłuższy od streszczenia redakcji, zawiera prefiks „(CNN)” i pomija ważną informację z dalszej części artykułu (sprzeciw Izraela i USA jest w 6. zdaniu). Mimo to pokrywa się z wieloma słowami streszczenia, bo kluczowe informacje są na początku artykułu (lead bias).

In [5]:
lead3_r2 = round(wynik["rouge2"] * 100, 2)
sufit_r2 = 21.28
roznica = sufit_r2 - lead3_r2

pd.DataFrame({
    "próg ROUGE-2": {
        "minimum (pobić Lead-3)": lead3_r2,
        "cel (odrobić 40% różnicy)": round(lead3_r2 + 0.4 * roznica, 2),
        "sufit (bart-large-cnn)": sufit_r2,
    }
})


,próg ROUGE-2
minimum (pobić Lead-3),17.49
cel (odrobić 40% różnicy),19.01
sufit (bart-large-cnn),21.28


## Z5. Metryka sukcesu

**Metryka główna:** ROUGE-2, bo liczy pokrycie pary słów,
więc lepiej niż ROUGE-1 sprawdza, czy model tworzy sensowne frazy.

**Progi:**
- minimum: ROUGE-2 > 17.49 
- cel: ROUGE-2 ≥ 19.01 (odrobienie 40% różnicy między Lead-3 a bart-large-cnn)
- sufit: ROUGE-2 = 21.28 (model ROUGE-2, nie mój wynik)

**Metryki dodatkowe:**
1. ROUGE-2 osobno dla krótkich i dlugich artykułów (pytanie pomocnicze 1: czy nie szkodzi)
2. Z której części artykułu pochodzą słowa streszczenia (pytanie pomocnicze 2: lead bias)
3. BERTScore, który mierzy podobieństwo znaczeniowe, a nie tylko wspólne słowa